# MTPC — Full training on Google Colab (Phase 0 → 1 → 2)

Re-trains the byT5-small MTPC models **from scratch** on a GPU runtime, then runs a sanity check.

### Trains WITHOUT `cheat` (on purpose)
`cheat` feeds the whole answer into byT5's **bidirectional** encoder; with un-masked cross-attention
the decoder just **copies** the answer it should predict — low loss in training, **byte-salad at
inference**. Standard SFT (cheat off) masks the answer in the encoder, so the model learns to predict.

Run cells **1 → 8** in order. First set *Runtime ▸ Change runtime type ▸ GPU* (A100 ideal; T4 works but
slower and needs a small batch on the CP/HMM phases).

## 1. Mount Drive & clone the repo

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout
      or "!! No GPU — set Runtime > Change runtime type > GPU, then re-run.")
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/MTPC'          # change if you like
REPO_URL    = 'https://github.com/lorenzoAllegrini/MTPC.git'
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} "{PROJECT_DIR}"
else:
    !cd "{PROJECT_DIR}" && git pull
%cd {PROJECT_DIR}

## 2. Install dependencies
Colab already ships a compatible `torch` (we don't touch it). If you later get an import error,
pin the known-good versions: `pip install transformers==5.8.0 peft==0.19.1`.

In [ ]:
!pip install -q -U transformers peft datasets accelerate

## 3. Configuration
The **CP/HMM** heads are memory-heavy (`ranks*window*vocab` outputs → `[B, L, ranks, window, vocab]`
activations). Defaults below are **T4-safe**. On an **A100** raise `BATCH_SIZE` to 8–16 for speed.
If you hit CUDA OOM on the CP/HMM cells, lower `BATCH_SIZE` to 1 and/or `MAX_LEN` to 512.

In [ ]:
MODEL_ID    = "google/byt5-small"
WINDOW_SIZE = 6
RANKS       = 32
MAX_SAMPLES = 20000     # flan_v2 examples after filtering (more = better, slower)
MAX_LEN     = 1024      # bytes per example (padded to this length)
BATCH_SIZE  = 2         # T4-safe default; A100: 8-16
# cheat stays OFF (no --cheat anywhere) -> no encoder answer-leakage.

## 4. Phase 0 + 1 + 2 — backbone SFT, FF warm-up, FF joint
Trains the shared backbone + the Feed-Forward draft, and produces the **verifier** backbone
(`byt5_standard_lora_phase0`). First run downloads & tokenizes Tülu-3 (slow; cached to Drive).

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --head ff "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 false --skip_phase_1 false --skip_phase_2 false")
print(cmd)
!{cmd}

## 5. Phase 2 — CP head
Reuses the Phase-0 backbone, transfers the Phase-1 FF emissions into the CP circuit, joint-trains it.

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --head cp "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 true --skip_phase_1 true --skip_phase_2 false")
print(cmd)
!{cmd}

## 6. (optional) Phase 2 — HMM head

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --head hmm "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 true --skip_phase_1 true --skip_phase_2 false")
print(cmd)
!{cmd}

## 7. Sanity check — honest (non-cheat) prediction
With cheat off, the argmax should be a real *prediction* (NOT a verbatim copy of the ground truth)
and the per-step loss should be well below the random baseline. If argmax == ground truth exactly,
cheat leaked somewhere.

In [ ]:
cmd = f"cd src && python inference.py --head cp --window_size {WINDOW_SIZE} --num_samples 5"
print(cmd)
!{cmd}

## 8. Done
Saved on Drive under `MTPC/saved_models/`:
- `byt5_standard_lora_phase0/` → verifier backbone
- `mtp_head_{ff,cp,hmm}_w6_final.pth` + `lora_{ff,cp,hmm}_w6/` → draft heads + adapters

Then `git pull` locally and run `speculative_inference_testing.R` (already set to `CHEAT = FALSE`).